# Notebook 13 — Validación cruzada contra cartografía oficial INVEMAR 1:25.000

Calcula una segunda validación independiente de la clasificación por umbrales del proyecto, esta vez contra la cartografía oficial de manglares de Colombia a escala 1:25.000 publicada por el Instituto de Investigaciones Marinas y Costeras (INVEMAR, 2020) ---disponible mediante el servicio ArcGIS REST `SIGMA/MANGLARES_COLOMBIA`---, y a su vez compara las dos cartografías globales y oficiales entre sí (ESA WorldCover v200 vs INVEMAR 1:25.000) sobre el AOI acotado SFF + VPI Salamanca. Esta segunda validación complementa la realizada en el notebook `10_validacion_extendida.ipynb` contra WorldCover (F1 = 0,548) y ofrece la referencia nacional oficial como contrapeso al producto cartográfico global.

**Insumos:**
- `data/validation/invemar_manglar_25k.geojson` (1000 polígonos, 31.912 ha sobre la región CGSM)
- `data/validation/gmw/worldcover_mangrove_2021_cgsm.tif` (raster WorldCover ya descargado)
- `data/raw/cgsm_aoi_acotado_4326.geojson`

**Productos:**
- `outputs/tables/validacion_invemar_acotado.csv`
- `outputs/tables/comparacion_cartografias_acotado.csv` (matriz de las tres comparaciones)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.mask import mask as rio_mask

ROOT = Path('..').resolve()
AOI_PATH = ROOT / 'data' / 'raw' / 'cgsm_aoi_acotado_4326.geojson'
INV_PATH = ROOT / 'data' / 'validation' / 'invemar_manglar_25k.geojson'
WC_PATH  = ROOT / 'data' / 'validation' / 'gmw' / 'worldcover_mangrove_2021_cgsm.tif'
OUT_TAB  = ROOT / 'outputs' / 'tables'
OUT_TAB.mkdir(parents=True, exist_ok=True)

for p in [AOI_PATH, INV_PATH, WC_PATH]:
    print(f'{"✓" if p.exists() else "✗"}  {p.name}')

✓  cgsm_aoi_acotado_4326.geojson
✓  invemar_manglar_25k.geojson
✓  worldcover_mangrove_2021_cgsm.tif


## 1. Cargar AOI, INVEMAR y WorldCover

WorldCover define la grilla de referencia (25 m, EPSG:4326, recortado al AOI acotado). INVEMAR se rasteriza sobre esa misma grilla para comparación pixel a pixel.

In [2]:
aoi = gpd.read_file(AOI_PATH).to_crs(4326)
inv = gpd.read_file(INV_PATH).to_crs(4326)

# Filtrar INVEMAR al AOI acotado
inv_aoi = inv[inv.intersects(aoi.geometry.union_all())].copy()
print(f'INVEMAR total: {len(inv)} polígonos, {inv.AREA_HA.sum():.1f} ha')
print(f'INVEMAR ∩ AOI acotado: {len(inv_aoi)} polígonos, {inv_aoi.AREA_HA.sum():.1f} ha')

# Cargar WorldCover y obtener su grilla
with rasterio.open(WC_PATH) as src:
    wc_data, wc_transform = rio_mask(src, aoi.geometry, crop=True)
    wc_data = (wc_data[0] > 0).astype(np.uint8)
    wc_meta = src.meta.copy()
    wc_meta.update({'height': wc_data.shape[0], 'width': wc_data.shape[1],
                    'transform': wc_transform, 'dtype': 'uint8'})

print(f'\nWorldCover dentro del AOI:')
print(f'  Grid: {wc_data.shape}, transform pixel: {wc_transform.a:.6f}°')
print(f'  Manglar WorldCover: {wc_data.sum():,} píxeles ({wc_data.sum() * 25 * 25 / 1e6:.1f} km² aprox)')

INVEMAR total: 1000 polígonos, 31912.3 ha
INVEMAR ∩ AOI acotado: 529 polígonos, 29671.2 ha

WorldCover dentro del AOI:
  Grid: (2326, 2371), transform pixel: 0.000225°
  Manglar WorldCover: 413,304 píxeles (258.3 km² aprox)


## 2. Rasterizar INVEMAR sobre la grilla de WorldCover

Cada polígono INVEMAR aporta 1 a los píxeles que cubre; los píxeles no cubiertos quedan en 0. La rasterización se hace sobre la misma grilla del WorldCover para que la confusion matrix sea pixel a pixel exacta.

In [3]:
shapes_inv = [(geom, 1) for geom in inv_aoi.geometry if geom.is_valid]
inv_data = rasterize(shapes_inv, out_shape=wc_data.shape, transform=wc_transform,
                     fill=0, dtype=np.uint8)

# También necesitamos una máscara del AOI para limitar el análisis a píxeles dentro
from rasterio.features import geometry_mask
aoi_mask = ~geometry_mask([g for g in aoi.geometry],
                          out_shape=wc_data.shape, transform=wc_transform, invert=False)

print(f'INVEMAR rasterizado:')
print(f'  Manglar INVEMAR: {inv_data.sum():,} píxeles ({inv_data.sum() * 25 * 25 / 1e6:.1f} km²)')
print(f'  Píxeles dentro del AOI: {aoi_mask.sum():,}')
print(f'  Manglar INVEMAR dentro AOI: {(inv_data & aoi_mask).sum():,} píxeles')

INVEMAR rasterizado:
  Manglar INVEMAR: 41,105 píxeles (25.7 km²)
  Píxeles dentro del AOI: 1,370,916
  Manglar INVEMAR dentro AOI: 34,981 píxeles


In [10]:
from shapely.validation import make_valid

# Limpiar geometrías inválidas
inv_aoi_clean = inv_aoi.copy()
inv_aoi_clean['geometry'] = inv_aoi_clean.geometry.apply(make_valid)
inv_aoi_clean = inv_aoi_clean[inv_aoi_clean.geometry.is_valid].reset_index(drop=True)
print(f'Después de make_valid: {len(inv_aoi_clean)} polígonos válidos')

# Re-rasterizar con all_touched=True
shapes_inv = [(geom, 1) for geom in inv_aoi_clean.geometry]
inv_data = rasterize(shapes_inv, out_shape=wc_data.shape, transform=wc_transform,
                     fill=0, dtype=np.uint8, all_touched=True)
print(f'\nINVEMAR rasterizado (fix):')
print(f'  Manglar INVEMAR: {inv_data.sum():,} píxeles')
print(f'  Área aproximada: {inv_data.sum() * 25 * 25 / 1e6:.1f} km²')
print(f'  Esperado: ~{inv_aoi_clean.AREA_HA.sum() / 100:.1f} km²')

Después de make_valid: 529 polígonos válidos

INVEMAR rasterizado (fix):
  Manglar INVEMAR: 564,729 píxeles
  Área aproximada: 353.0 km²
  Esperado: ~296.7 km²


## 3. Matriz de confusión: INVEMAR vs ESA WorldCover

Compara las dos cartografías de referencia entre sí sobre el AOI acotado, lo que ofrece una medida directa de su acuerdo independientemente del clasificador del proyecto.

In [11]:
def confusion_matrix_metrics(y_true, y_pred, mask):
    """Devuelve TP/FP/FN/TN y métricas, restringido al mask."""
    y_t = y_true[mask].astype(bool)
    y_p = y_pred[mask].astype(bool)
    tp = int((y_t & y_p).sum())
    fp = int((~y_t & y_p).sum())
    fn = int((y_t & ~y_p).sum())
    tn = int((~y_t & ~y_p).sum())
    oa  = (tp + tn) / max(tp + tn + fp + fn, 1)
    pre = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1  = 2 * pre * rec / max(pre + rec, 1e-9)
    spec= tn / max(tn + fp, 1)
    return dict(TP=tp, FP=fp, FN=fn, TN=tn,
                OA=round(oa, 4), Precision=round(pre, 4),
                Recall=round(rec, 4), F1=round(f1, 4),
                Specificity=round(spec, 4))

# INVEMAR (referencia) vs WorldCover (predicción)
m1 = confusion_matrix_metrics(inv_data, wc_data, aoi_mask)
print('=== INVEMAR (truth) vs ESA WorldCover (pred) ===')
for k, v in m1.items():
    print(f'  {k:12s}: {v}')

=== INVEMAR (truth) vs ESA WorldCover (pred) ===
  TP          : 344441
  FP          : 68863
  FN          : 69389
  TN          : 888223
  OA          : 0.8992
  Precision   : 0.8334
  Recall      : 0.8323
  F1          : 0.8329
  Specificity : 0.928


## 4. Descargar clasificación por umbrales desde GEE

Reconstruye la clasificación por umbrales (NDVI > 0,70 ∧ elevación < 10 m ∧ distancia al agua < 3 km) sobre el AOI acotado y la exporta como TIF a 25 m para comparación local con INVEMAR y WorldCover sobre la misma grilla.

In [12]:
import ee, geemap

try:
    ee.Initialize(project='basic-buttress-338101')
except Exception:
    import google.auth
    creds, _ = google.auth.default()
    ee.Initialize(credentials=creds, project='basic-buttress-338101')

aoi_ee = ee.Geometry(aoi.geometry.union_all().__geo_interface__)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi_ee).filterDate('2020-01-01', '2020-12-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))
ndvi = s2.median().normalizedDifference(['B8','B4']).clip(aoi_ee)
elev = ee.Image('USGS/SRTMGL1_003').clip(aoi_ee).lt(10)
jrc  = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').clip(aoi_ee)
near = jrc.gt(30).fastDistanceTransform().sqrt().multiply(30).lt(3000)
clf  = ndvi.gt(0.70).And(elev).And(near).rename('clf').unmask(0).clip(aoi_ee)

CLF_TIF = ROOT / 'data' / 'validation' / 'clasificacion_umbrales_2020_acotado.tif'
if CLF_TIF.exists():
    CLF_TIF.unlink()
print('Exportando clasificación umbrales a TIF (25 m)...')
geemap.ee_export_image(
    clf.toByte(),
    filename=str(CLF_TIF),
    region=aoi_ee, scale=25, crs='EPSG:4326',
)
print(f'OK: {CLF_TIF.name}  ({CLF_TIF.stat().st_size / 1024:.0f} KB)' if CLF_TIF.exists()
      else 'FALLO descarga')

Exportando clasificación umbrales a TIF (25 m)...
Generating URL ...
Please wait ...
Data downloaded to /home/rstudio/work/proyecto-cgsm/data/validation/clasificacion_umbrales_2020_acotado.tif
OK: clasificacion_umbrales_2020_acotado.tif  (73 KB)


In [6]:
# Cargar clasificación y alinear a la grilla de WorldCover
from rasterio.warp import reproject, Resampling as Rs

with rasterio.open(CLF_TIF) as src:
    clf_data_native = src.read(1)
    clf_native_transform = src.transform
    clf_native_crs = src.crs

# Reproyectar la clasificación a la grilla exacta de WorldCover
clf_data = np.zeros(wc_data.shape, dtype=np.uint8)
reproject(
    source=clf_data_native, destination=clf_data,
    src_transform=clf_native_transform, src_crs=clf_native_crs,
    dst_transform=wc_transform, dst_crs='EPSG:4326',
    resampling=Rs.nearest,
)
clf_data = (clf_data > 0).astype(np.uint8)
print(f'Clasificación umbrales alineada:')
print(f'  Manglar clasif: {clf_data.sum():,} píxeles ({clf_data.sum() * 25 * 25 / 1e6:.1f} km²)')

Clasificación umbrales alineada:
  Manglar clasif: 234,490 píxeles (146.6 km²)


## 5. Matriz de confusión: INVEMAR vs clasificación por umbrales del proyecto

In [13]:
m2 = confusion_matrix_metrics(inv_data, clf_data, aoi_mask)
print('=== INVEMAR (truth) vs clasificación umbrales (pred) ===')
for k, v in m2.items():
    print(f'  {k:12s}: {v}')

=== INVEMAR (truth) vs clasificación umbrales (pred) ===
  TP          : 188097
  FP          : 43777
  FN          : 225733
  TN          : 913309
  OA          : 0.8034
  Precision   : 0.8112
  Recall      : 0.4545
  F1          : 0.5826
  Specificity : 0.9543


## 6. Matriz de confusión: WorldCover vs clasificación (replica notebook 10 sobre raster local)

In [14]:
m3 = confusion_matrix_metrics(wc_data, clf_data, aoi_mask)
print('=== ESA WorldCover (truth) vs clasificación umbrales (pred) ===')
for k, v in m3.items():
    print(f'  {k:12s}: {v}')

=== ESA WorldCover (truth) vs clasificación umbrales (pred) ===
  TP          : 177826
  FP          : 54048
  FN          : 235478
  TN          : 903564
  OA          : 0.7888
  Precision   : 0.7669
  Recall      : 0.4303
  F1          : 0.5512
  Specificity : 0.9436


## 7. Resumen consolidado de las tres comparaciones

In [15]:
import pandas as pd

df = pd.DataFrame([
    {'comparacion': 'INVEMAR vs WorldCover',           **m1},
    {'comparacion': 'INVEMAR vs clasificacion',        **m2},
    {'comparacion': 'WorldCover vs clasificacion',     **m3},
])
df.to_csv(OUT_TAB / 'comparacion_cartografias_acotado.csv', index=False)
print(df.to_string(index=False))
print(f'\nGuardado: {OUT_TAB / "comparacion_cartografias_acotado.csv"}')

# Tabla específica de validación contra INVEMAR para el informe
df_inv = pd.DataFrame([
    {'metrica': 'TP', 'valor': m2['TP']},
    {'metrica': 'FP', 'valor': m2['FP']},
    {'metrica': 'FN', 'valor': m2['FN']},
    {'metrica': 'TN', 'valor': m2['TN']},
    {'metrica': 'OA', 'valor': m2['OA']},
    {'metrica': 'Precision', 'valor': m2['Precision']},
    {'metrica': 'Recall', 'valor': m2['Recall']},
    {'metrica': 'F1', 'valor': m2['F1']},
    {'metrica': 'Specificity', 'valor': m2['Specificity']},
])
df_inv.to_csv(OUT_TAB / 'validacion_invemar_acotado.csv', index=False)
print(f'Guardado: {OUT_TAB / "validacion_invemar_acotado.csv"}')

                comparacion     TP    FP     FN     TN     OA  Precision  Recall     F1  Specificity
      INVEMAR vs WorldCover 344441 68863  69389 888223 0.8992     0.8334  0.8323 0.8329       0.9280
   INVEMAR vs clasificacion 188097 43777 225733 913309 0.8034     0.8112  0.4545 0.5826       0.9543
WorldCover vs clasificacion 177826 54048 235478 903564 0.7888     0.7669  0.4303 0.5512       0.9436

Guardado: /home/rstudio/work/proyecto-cgsm/outputs/tables/comparacion_cartografias_acotado.csv
Guardado: /home/rstudio/work/proyecto-cgsm/outputs/tables/validacion_invemar_acotado.csv


## 8. Interpretación esperada

Si los F1 de las dos comparaciones del clasificador contra ambas cartografías ---WorldCover (notebook 10, F1 = 0,548) e INVEMAR (este notebook)--- son similares, queda confirmado que la metodología del proyecto produce una clasificación consistente independientemente de la cartografía de referencia. Si los F1 difieren marcadamente, la fuente de la discrepancia podría ser: WorldCover usa un clasificador supervisado a 10 m con criterio inclusivo y captura manglar joven o degradado; INVEMAR aplica criterios más exigentes de delimitación 1:25.000 y puede excluir parches pequeños o de transición.

La comparación adicional INVEMAR vs WorldCover ofrece una medida del acuerdo entre ambas cartografías y permite contextualizar los F1 del proyecto: si las dos cartografías oficiales tampoco coinciden perfectamente entre sí, los F1 del clasificador deben leerse en ese marco de incertidumbre estructural y no como medida absoluta de calidad.